# FIS Input Preparation v1

Prepare the raster inputs used by `FIS_v1.ipynb`.

This notebook is scoped only to the FIS/DoD uncertainty workflow. It creates:

- slope from the post-event DEM
- slope from the pre-event DEM
- a uniform QL2 LiDAR point-density raster in points per square metre
- NLCD canopy cover resampled to the DEM grid

All paths are centralized in the setup cell and outputs are written to a dedicated `fis_inputs` folder under the Thomas data directory.

Recommended kernel/environment: `ml_debris` for the canopy download cell (`pygeohydro` and `rioxarray`). The slope step has a NumPy fallback if `richdem` is unavailable.


In [ ]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

BASE_DIR = Path("/mnt/c/Users/amehedi/Downloads/thomas")
DEM_POST_PATH = BASE_DIR / "dem_post.tif"
DEM_PRE_PATH = BASE_DIR / "dem_pre.tif"
AOI_PATH = BASE_DIR / "montecito_aoi.shp"

OUT_DIR = BASE_DIR / "fis_inputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SLOPE_POST_PATH = OUT_DIR / "slope_10m_post.tif"
SLOPE_PRE_PATH = OUT_DIR / "slope_10m_pre.tif"
POINT_DENSITY_PATH = OUT_DIR / "point_density_10m.tif"
CANOPY_30M_PATH = OUT_DIR / "canopy_2019_30m.tif"
CANOPY_10M_PATH = OUT_DIR / "canopy_2019_10m.tif"

NODATA = -9999.0
POINT_DENSITY_PTS_M2 = 1.0  # USGS QL2 minimum nominal pulse density

for label, path in {
    "post DEM": DEM_POST_PATH,
    "pre DEM": DEM_PRE_PATH,
    "AOI": AOI_PATH,
}.items():
    print(f"{label}: {path} exists={path.exists()}")

print("output folder:", OUT_DIR)

post DEM: /mnt/c/Users/amehedi/Downloads/thomas/dem_post.tif exists=True
pre DEM: /mnt/c/Users/amehedi/Downloads/thomas/dem_pre.tif exists=True
AOI: /mnt/c/Users/amehedi/Downloads/thomas/montecito_aoi.shp exists=True
output folder: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs


## 1. Slope Rasters

Compute slope in degrees for both DEMs. The preferred method is `richdem`; if it is not installed, the fallback uses a NumPy gradient and `atan(rise/run)`.


In [2]:
def write_float_raster(path, array, profile, nodata=NODATA):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    data = np.asarray(array, dtype="float32")
    out = np.where(np.isfinite(data), data, nodata).astype("float32")

    meta = profile.copy()
    meta.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=nodata,
        compress="deflate",
    )
    if not meta.get("tiled", False):
        meta.pop("blockxsize", None)
        meta.pop("blockysize", None)

    with rasterio.open(path, "w", **meta) as dst:
        dst.write(out, 1)
    return path


def slope_with_numpy(dem, profile):
    transform = profile["transform"]
    dx = abs(float(transform.a))
    dy = abs(float(transform.e))
    dz_dy, dz_dx = np.gradient(dem, dy, dx)
    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    return np.degrees(slope_rad)


def compute_slope_degrees(dem_path, out_path):
    with rasterio.open(dem_path) as src:
        dem = src.read(1).astype("float64")
        profile = src.profile.copy()
        nodata = src.nodata

    invalid = ~np.isfinite(dem)
    if nodata is not None:
        invalid |= np.isclose(dem, nodata)
    dem[invalid] = np.nan

    try:
        import richdem as rd

        dem_rd = rd.rdarray(dem, no_data=np.nan)
        dem_rd.geotransform = profile["transform"].to_gdal()
        slope = np.asarray(rd.TerrainAttribute(dem_rd, attrib="slope_degrees"), dtype="float32")
        method = "richdem"
    except Exception as exc:
        print(f"richdem unavailable for {dem_path.name}; using NumPy gradient fallback: {exc}")
        slope = slope_with_numpy(dem, profile).astype("float32")
        method = "numpy_gradient"

    slope[invalid] = np.nan
    write_float_raster(out_path, slope, profile)
    print(f"saved slope ({method}): {out_path}")
    print("  min/mean/max:", float(np.nanmin(slope)), float(np.nanmean(slope)), float(np.nanmax(slope)))


compute_slope_degrees(DEM_POST_PATH, SLOPE_POST_PATH)
compute_slope_degrees(DEM_PRE_PATH, SLOPE_PRE_PATH)


/home/abdullah/miniconda3/envs/ml_debris/lib/python3.10/site-packages/richdem/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources

A Slope calculation (degrees)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.00785352===================== ] (99% - 0.0s - 1 threads)


saved slope (richdem): /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/slope_10m_post.tif
  min/mean/max: 0.006797215435653925 22.119726181030273 63.32152557373047
saved slope (richdem): /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/slope_10m_pre.tif
  min/mean/max: 0.007041563279926777 22.84827423095703 62.51799392700195



A Slope calculation (degrees)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.00787303===================== ] (99% - 0.0s - 1 threads)


## 2. Point Density Raster

Create a uniform point-density raster in `pts/m2` on the post-event DEM grid. The FIS notebook uses this as the LiDAR point-density input.


In [3]:
with rasterio.open(DEM_POST_PATH) as src:
    ref = src.read(1).astype("float32")
    profile = src.profile.copy()
    nodata = src.nodata

invalid = ~np.isfinite(ref)
if nodata is not None:
    invalid |= np.isclose(ref, nodata)

point_density = np.full(ref.shape, np.nan, dtype="float32")
point_density[~invalid] = POINT_DENSITY_PTS_M2

write_float_raster(POINT_DENSITY_PATH, point_density, profile)
print("saved point density:", POINT_DENSITY_PATH)
print("density pts/m2:", POINT_DENSITY_PTS_M2)


saved point density: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/point_density_10m.tif
density pts/m2: 1.0


## 3. NLCD Canopy Cover

Download NLCD 2019 canopy cover for the AOI using `pygeohydro`, then save the 30 m source raster. If the 30 m raster already exists, this cell reuses it.


In [4]:
def download_canopy_30m():
    try:
        import geopandas as gpd
        import pygeohydro as gh
        import rioxarray  # noqa: F401 - activates the .rio accessor
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "The canopy-download cell needs pygeohydro and rioxarray. "
            "Use the ml_debris conda environment, install those packages, "
            f"or place an existing canopy raster at {CANOPY_30M_PATH} or {CANOPY_10M_PATH}."
        ) from exc

    if not AOI_PATH.exists():
        raise FileNotFoundError(f"Missing AOI shapefile: {AOI_PATH}")

    geom = gpd.read_file(AOI_PATH).to_crs("EPSG:4326")
    canopy = gh.nlcd_bygeom(geom, resolution=30, years={"canopy": [2019]})

    if isinstance(canopy, dict):
        canopy_ds = next(iter(canopy.values()))
    else:
        canopy_ds = canopy[0]

    data_vars = list(canopy_ds.data_vars)
    canopy_var = next((name for name in data_vars if "canopy" in name.lower()), data_vars[0])
    canopy_da = canopy_ds[canopy_var]

    canopy_da.rio.to_raster(CANOPY_30M_PATH)
    print("saved canopy 30 m:", CANOPY_30M_PATH)
    print("canopy variable:", canopy_var)


if CANOPY_30M_PATH.exists():
    print("using existing canopy 30 m:", CANOPY_30M_PATH)
elif CANOPY_10M_PATH.exists():
    print("canopy 10 m already exists, skipping 30 m download:", CANOPY_10M_PATH)
else:
    download_canopy_30m()


using existing canopy 30 m: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/canopy_2019_30m.tif


## 4. Align Canopy To DEM Grid

Resample the 30 m canopy raster to the 10 m post-event DEM grid. The result is used by `FIS_v1.ipynb`.


In [5]:
def align_to_reference(src_path, ref_path, out_path, resampling=Resampling.bilinear, nodata=NODATA):
    with rasterio.open(ref_path) as ref, rasterio.open(src_path) as src:
        ref_arr = ref.read(1)
        ref_invalid = ~np.isfinite(ref_arr)
        if ref.nodata is not None:
            ref_invalid |= np.isclose(ref_arr, ref.nodata)

        out = np.full((ref.height, ref.width), nodata, dtype="float32")

        reproject(
            source=rasterio.band(src, 1),
            destination=out,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref.transform,
            dst_crs=ref.crs,
            dst_nodata=nodata,
            resampling=resampling,
        )

        out[ref_invalid] = nodata
        profile = ref.profile.copy()

    write_float_raster(out_path, np.where(np.isclose(out, nodata), np.nan, out), profile, nodata=nodata)
    return out_path


align_to_reference(CANOPY_30M_PATH, DEM_POST_PATH, CANOPY_10M_PATH)
print("saved canopy 10 m:", CANOPY_10M_PATH)


saved canopy 10 m: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/canopy_2019_10m.tif


## 5. Check Outputs


In [6]:
for path in [SLOPE_POST_PATH, SLOPE_PRE_PATH, POINT_DENSITY_PATH, CANOPY_30M_PATH, CANOPY_10M_PATH]:
    if not path.exists():
        print("missing:", path)
        continue
    with rasterio.open(path) as src:
        print(path.name)
        print("  path:", path)
        print("  shape:", (src.height, src.width), "crs:", src.crs, "res:", src.res, "nodata:", src.nodata)


slope_10m_post.tif
  path: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/slope_10m_post.tif
  shape: (835, 716) crs: EPSG:32611 res: (10.0, 10.0) nodata: -9999.0
slope_10m_pre.tif
  path: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/slope_10m_pre.tif
  shape: (835, 716) crs: EPSG:32611 res: (10.0, 10.0) nodata: -9999.0
point_density_10m.tif
  path: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/point_density_10m.tif
  shape: (835, 716) crs: EPSG:32611 res: (10.0, 10.0) nodata: -9999.0
canopy_2019_30m.tif
  path: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/canopy_2019_30m.tif
  shape: (285, 238) crs: EPSG:4326 res: (0.0003257100840336201, 0.00026959999999999034) nodata: nan
canopy_2019_10m.tif
  path: /mnt/c/Users/amehedi/Downloads/thomas/fis_inputs/canopy_2019_10m.tif
  shape: (835, 716) crs: EPSG:32611 res: (10.0, 10.0) nodata: -9999.0
